**Introdution to the Project :**

A fixed weight Alexandria map GPS . This project provides a comparative study of two types of search algorithms ,uninformed search [BFS] versus informed search [A*] to see which one is better for GPS applications.

**Important notes :**

In the heuristic file you can see the (X,Y) columns , those are the real coordinates of the Alexandria places (obtained from Google maps). The coordinates are used in the Euclidean distance formula [math.sqrt((x2 - x1)**2 + (y2 - y1)2)111] to give us the final heuristic value. In the Euclidean distance formula you can see i multiplied the equation by 111 , this is because when coordinates are calculated it give you a tiny heuristic value for example 0.043 , and when the heuristic value is this tiny the code ignores it and treats it like its nothing , basically it treats it like the BFS with no heuristic . thats why i multiplied it by 111. In the comparison table you can clearly see that the nodes expanded by the A is less than the BFS , thats mean the A is better at searching for the goal than the BFS. You can run the Astar_with_gui file and choose a start point and a goal to see the A road map and the estimated travel time and the traveled distance in kilometers.

**YOU CAN CHECK THE GITHUB LINK PROVIDED TO SEE THE PROJECT WITH THE GUI.**

In [1]:
import csv
import math
from collections import deque
import os

In [2]:
h_path ='/kaggle/input/datasets/bahaaldinalhadi/alexandria-map/Heuristic_data.csv'
r_path ='/kaggle/input/datasets/bahaaldinalhadi/alexandria-map/Roads.csv'

In [3]:
# =============================Loading the Roads file====================================
def load_roads(roads_file):
    rgraph = {}
    with open(roads_file, mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            s = row["NodeA"]
            e = row["NodeB"]
            dist = float(row["Distance_Km"])
            speed = int(row["Speed_Limit"])
            travel_time = (dist / speed) * 60
            if s not in rgraph: rgraph[s] = []
            if e not in rgraph: rgraph[e] = []
            rgraph[s].append({"node": e, "time": travel_time, "distance": dist})
            rgraph[e].append({"node": s, "time": travel_time, "distance": dist})
    return rgraph

In [4]:
#==============================Loading the heuristic==========================

def load_heuristics(heuristic_file):
    hgraph = {}
    with open(heuristic_file, mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            hgraph[row["Name"]] = (float(row["X_Coord"]), float(row["Y_Coord"]))
    return hgraph

In [5]:
#============================Implementing The Euclidean distance formula to get the heuristic value======================
def heuristic_value(NodeA, NodeB, coords):
    if NodeA not in coords or NodeB not in coords:
        return 0
    x1, y1 = coords[NodeA]
    x2, y2 = coords[NodeB]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)*111

In [6]:
# =========================A star========================================================
def Astar(graph, start, goal, coords):
    expanded_nodes = 0 
    queue = [[(start, 0, 0)]]
    visited = []

    while queue:
        queue.sort(key=lambda path: path[-1][1] + heuristic_value(path[-1][0], goal, coords))
        
        path = queue.pop(0)
        node, g, d = path[-1]
        
        if node in visited: 
            continue
        expanded_nodes += 1
        visited.append(node)
        
        if node == goal:
            station_names = [step[0] for step in path]
            return station_names, g, d, expanded_nodes
        
        for neighbor in graph.get(node, []):
            if neighbor['node'] not in visited:
                new_path = list(path)
                new_path.append((neighbor['node'], g + neighbor['time'], d + neighbor['distance']))
                queue.append(new_path)
    return None, 0, 0, expanded_nodes

In [7]:
def bfs_path(graph, start, goal):
    visited = {start}
    queue = deque([[start]])
    expanded_nodes = 0
    while queue:
        path = queue.popleft()
        current_node = path[-1]
        expanded_nodes += 1
        if current_node == goal:
            return path, expanded_nodes
        if current_node in graph:
            for neighbor_dict in graph[current_node]:
                neighbor = neighbor_dict['node']
                if neighbor not in visited:
                    visited.add(neighbor)
                    new_path = list(path)
                    new_path.append(neighbor)
                    queue.append(new_path)
    return None, expanded_nodes

In [8]:
alex_roads = load_roads(r_path)
alex_coords = load_heuristics(h_path)

In [9]:
queries = [
    ("Alagami", "Abo_Qeer"),
    ("Miami", "Almanshia"),
    ("Sidi_Gaber", "Sanstefano"),
    ("Almalaha", "Kafr_abdo"),
    ("Bacos", "Alibrahimia"),
    ("Miami", "Smouha")
]

comparison_data = []

print("===  DETAILED NAVIGATION RESULTS ===\n")
for start_node, end_node in queries:
    # Get results from both algorithms
    a_path, a_time, a_dist, a_nodes = Astar(alex_roads, start_node, end_node, alex_coords)
    b_path, b_nodes = bfs_path(alex_roads, start_node, end_node)
    
    if a_path and b_path:
        print(f"Query: From {start_node} to {end_node}")
        
        # Print A* Result
        print(f"  [A* Path]  : {' -> '.join(a_path)}")
        
        # Print BFS Result
        print(f"  [BFS Path] : {' -> '.join(b_path)}")
        
        print("-" * 50)
        comparison_data.append([f"{start_node}->{end_node}", a_nodes, b_nodes, f"{a_dist:.2f} km"])
        
# -------------------------------------COMPARISON TABLE----------------------
print("\n=== COMPARISON TABLE ===")
print("-" * 80)
print(f"{'Route':<35} | {'A* Nodes':<12} | {'BFS Nodes':<12} | {'Distance'}")
print("-" * 80)
for row in comparison_data:
    print(f"{row[0]:<35} | {row[1]:<12} | {row[2]:<12} | {row[3]}")
print("-" * 80)

===  DETAILED NAVIGATION RESULTS ===

Query: From Alagami to Abo_Qeer
  [A* Path]  : Alagami -> Alwardiyan -> Almanshia -> Camp_Shizar -> Sporting -> Almosheer_st -> Stanli -> Gleem -> Sanstefano -> Almahroosa -> Mohammed_nageeb -> Miami -> Alasafra -> Almandara -> Almamoura -> Toson -> Abo_Qeer
  [BFS Path] : Alagami -> Alwardiyan -> Almanshia -> Camp_Shizar -> Sporting -> Almosheer_st -> Stanli -> Gleem -> Bacos -> Alsoyof -> FortyFiveSt_gibli -> Almalaha -> Almamoura -> Toson -> Abo_Qeer
--------------------------------------------------
Query: From Miami to Almanshia
  [A* Path]  : Miami -> Mohammed_nageeb -> Almahroosa -> Sanstefano -> Gleem -> Stanli -> Almosheer_st -> Sporting -> Camp_Shizar -> Almanshia
  [BFS Path] : Miami -> Mohammed_nageeb -> Almahroosa -> Sanstefano -> Gleem -> Stanli -> Almosheer_st -> Sporting -> Camp_Shizar -> Almanshia
--------------------------------------------------
Query: From Sidi_Gaber to Sanstefano
  [A* Path]  : Sidi_Gaber -> Almosheer_st -> Sta